## Initialisatie


### Imports


In [ ]:
import os
import ee
import pandas as pd
from dotenv import load_dotenv

from tools.Input_data.satellites import sentinel2_harmonized
from tools.Input_data.plot import PointType, corners_to_columns
from tools.Input_data.locaties import select_naturenreserves, RegionMode
from tools.extract_downloads.plot import estimate_extraction_workload ,build_imagecollection_stats, export_featurecollection_to_local, export_featurecollection_to_drive
from tools.data_cleansing.sentinel2 import add_local_cloud_cover, scl_cloud_removal_20m, scale_reflectance
from tools.calculate_spectral_indices.sentinel2 import (
    calc_bsi,
    calc_cire, 
    calc_evi ,
    calc_evi2,
    calc_fai,
    calc_gci,
    calc_gndvi,
    calc_ibi,
    calc_mndwi,
    calc_msi,
    calc_nbr,
    calc_nbr2,
    calc_ndbi,
    calc_ndmi,
    calc_ndre_b5,
    calc_ndre_b6,
    calc_ndre_b7,
    calc_ndsi,
    calc_ndti,
    calc_ndvi,
    calc_ndwi,
    calc_nirv,
    calc_osavi,
    calc_s2wi,
    calc_savi,
    calc_snow_brightness,
    calc_vari,
)


### Specialistische functies


In [ ]:
def join_unique_dates(series: pd.Series) -> str:
    """
    Zet een pandas Series met datumwaarden om naar één string
    met unieke, gesorteerde datums in ISO-formaat.

    Parameters
    ----------
    series : pd.Series
        Series met datumwaarden, bij voorkeur in het formaat YYYYMMDD.

    Returns
    -------
    str
        Een string met unieke datums, gescheiden door komma's,
        bijvoorbeeld: "2020-07-10, 2020-07-11".
    """
    dates = pd.to_datetime(series.astype(str), format="%Y%m%d", errors="coerce")
    dates = dates.dropna().drop_duplicates().sort_values()
    return ", ".join(dates.dt.strftime("%Y-%m-%d"))

### Notebook project instellen


In [ ]:
load_dotenv()

project_code_ee = os.getenv("PROJECT_CODE_EE")

if not project_code_ee:
    raise RuntimeError("PROJECT_CODE_EE not found")

try:
    ee.Initialize(project=project_code_ee)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=project_code_ee)

In [ ]:
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

### Standaard waarden project


In [ ]:
# Projectgebied
natuurgebieden = select_naturenreserves(["Nieuwkoopse Plassen & de Haeck"])
type_coverage: RegionMode = "bounds"
buffer_size_meters: int = 0

filter_date_start: str = "2020-07-10"
filter_date_end: str = "2020-07-12"
local_cloud_cover_threshold: int = 30
cloud_cover_bloat: int = 100

# onderzoekslocaties
inputdata_cord_sys: str = r"EPSG:28992"
inputdata_pq_plot_csv: str = r"Inputdata\PQ_coordinaat_datum_Sjoerd.csv"
cell_size_width: int = 10  # meters
cell_size_height: int = 10  # meters
rotation_deg: int = 30  # clockwise rotation
corner_type: PointType = (
    "center"  # "center", "top_left", "top_right", "bottom_right", "bottom_left"
)

bands_to_revieuw : list[str] = ["NDVI", "EVI"]
scale_output_raster: int = 10

## code


### Onderzoeksgebieden plotten


In [ ]:
locations_df = pd.read_csv(inputdata_pq_plot_csv)

unique_groups = locations_df.groupby(["PQ_nummer", "X", "Y"], as_index=False).agg(
    {"Datum": join_unique_dates}
)

In [ ]:
# Bereken de hoekcoördinaten voor elke rij
corner_cols = unique_groups.apply(
    corners_to_columns,
    x_col="X",
    y_col="Y",
    axis=1,
    width=cell_size_width,
    height=cell_size_height,
    rotation_deg=rotation_deg,
    point_type=corner_type,
)

# Voeg de nieuwe hoekkolommen toe aan de originele DataFrame
input_plots = pd.concat([unique_groups, corner_cols], axis=1)

### Satelliet database opzetten


In [ ]:
s2 = (
    ee.ImageCollection(sentinel2_harmonized)
    .filterDate(filter_date_start, filter_date_end)
    .filterBounds(natuurgebieden)
    .map(add_local_cloud_cover(natuurgebieden, coverage=type_coverage))
    .map(scl_cloud_removal_20m(remove_unclassified=False, bloat_n_meters=cloud_cover_bloat))
    .map(scale_reflectance)
    .map(calc_ndvi)
    .map(calc_evi)
)

### Kosten schatten

In [ ]:
estimate_extraction_workload(
    features = input_plots,
    image_collection= s2,
    band_names= bands_to_revieuw,
    scale= scale_output_raster,
    include_mean= True,
    include_median= True,
    include_percentiles= (25, 75),
    include_stddev= True,
    include_minmax = True,
    include_count= True,
    crs= "EPSG:28992",
)

### Resultaten berekenen

In [ ]:
resultaten = build_imagecollection_stats(
    features= input_plots,
    image_collection= s2,
    band_names= bands_to_revieuw,
    scale= scale_output_raster,
    crs= "EPSG:28992",
    
    feature_property_columns=["PQ_nummer", "Datum"],
    image_property_columns=["MGRS_TILE"],

    include_count=True,
    include_date=True,

    include_mean= True,
    include_median= True,
    include_percentiles = (25, 75),
    include_stddev = True,
    include_minmax = False,

)
resultaten["workload_estimate"]

In [ ]:
output_df = export_featurecollection_to_local(
    feature_collection=resultaten["result_feature_collection"],
    local_file_path=r"output/resultaten.csv",
    keep_geometry=True,
    file_format="csv"
)

In [ ]:
output_drive = export_featurecollection_to_drive(
    feature_collection=resultaten["result_feature_collection"],
    description="testrun",
    file_name_prefix="OutputresultatenSentinel2",
    folder="Remote_Sensing",
    file_format="CSV",
    selectors=resultaten["selectors"],
    include_geometry=False
)